<a href="https://colab.research.google.com/github/slowanimals/CSE-144-Final-Project/blob/main/v2_Logan_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSE 144 Final Project — Transfer Learning Challenge (v2)

**Task:** 100-class image classification using pretrained EfficientNet-B4 fine-tuned on a small custom dataset.

**Dataset:** 100 classes × ~10 training images each, 1000 unlabeled test images.

**Improvements over v1:**
- EfficientNet-B4 backbone (19M params, 380px) — stronger ImageNet features than B2
- `RandomResizedCrop` augmentation — more spatial variation per epoch
- MixUp training (α=0.2) — smoother decision boundaries, less memorization
- RandomErasing — hides random patches, forces robustness
- 6-pass Test-Time Augmentation — averages center crop, hflip, and 2 scale variants at inference
- Early stopping (patience=7) — stops when val acc plateaus

**To run on Google Colab:** Runtime → Change runtime type → **T4 GPU**, then run all cells.

In [ ]:
# ── Colab: set Kaggle credentials ─────────────────────────────────────────────
import os
import sys

ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    token = 'KGAT_2b9604682d0cbd9da91112e4b824acf9'  # replace if expired
    with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
        f.write(token)
    os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
    print('Kaggle credentials written.')
else:
    print('Running locally — skipping Colab credential setup.')

In [ ]:
# ── Install dependencies (uncomment if needed) ────────────────────────────────
# !pip -q install torch torchvision matplotlib tqdm pandas kagglehub

In [ ]:
# ── Imports & reproducibility ─────────────────────────────────────────────────
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights


def set_seed(seed: int = 42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: No GPU detected. Training will be very slow. '
          'On Colab: Runtime → Change runtime type → T4 GPU.')

## 1. Hyperparameters

All tunable knobs in one place. Edit these before running the training cells.

In [ ]:
NUM_CLASSES   = 100
IMG_SIZE      = 380        # EfficientNet-B4 native input size
BATCH_SIZE    = 32
NUM_WORKERS   = 0          # keep 0 for reproducibility; increase for speed on Colab
PHASE1_EPOCHS = 5          # warm-up: backbone frozen
PHASE2_EPOCHS = 25         # fine-tune: all layers
PATIENCE      = 7          # early stopping patience
MIXUP_ALPHA   = 0.2        # MixUp blend strength; set 0 to disable
CKPT_PATH     = './checkpoints/best_final_project_v2.pt'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'Training for up to {PHASE1_EPOCHS + PHASE2_EPOCHS} epochs '
      f'(early stop patience={PATIENCE})')

## 2. Download Dataset

In [ ]:
import kagglehub

path = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')
print('Path to competition files:', path)

train_dir = os.path.join(path, 'train')
test_dir  = os.path.join(path, 'test')
print('Train dir:', train_dir)
print('Test dir: ', test_dir)

## 3. Dataset Classes

**`NumericalImageFolder`** — overrides PyTorch's default alphabetical sort so folder `"10"` maps to label 10, not label 2.

**`TestDataset`** — loads unlabeled test images in numerical order and returns the filename alongside the tensor so we can build `submission.csv`.

In [ ]:
class NumericalImageFolder(ImageFolder):
    """ImageFolder sorted numerically: '0'→0, '1'→1, ..., '99'→99."""
    def find_classes(self, directory):
        classes = sorted(
            [d.name for d in os.scandir(directory) if d.is_dir()],
            key=lambda x: int(x)
        )
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx


class TestDataset(Dataset):
    """Loads test images in numerical order (0.jpg, 1.jpg, ..., 999.jpg)."""
    def __init__(self, test_dir, transform=None):
        self.test_dir  = test_dir
        self.transform = transform
        self.filenames = sorted(
            [f for f in os.listdir(test_dir) if f.lower().endswith('.jpg')],
            key=lambda x: int(os.path.splitext(x)[0])
        )

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.test_dir, self.filenames[idx])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.filenames[idx]


# Sanity-check label ordering
_tmp = NumericalImageFolder(train_dir, transform=transforms.ToTensor())
assert _tmp.class_to_idx['0'] == 0 and _tmp.class_to_idx['10'] == 10, \
    'Label ordering is wrong!'
print(f'Total training samples: {len(_tmp)}')
print(f"class_to_idx check: '0'→{_tmp.class_to_idx['0']}, '10'→{_tmp.class_to_idx['10']}")
del _tmp

## 4. Transforms & DataLoaders

**Train:** `RandomResizedCrop` + flips + rotation + color jitter + erasing — each epoch every image looks different, effectively multiplying the dataset size.

**Val/Test:** Resize slightly larger then `CenterCrop` — matches how B4 was evaluated during ImageNet pretraining.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

val_tf = transforms.Compose([
    transforms.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# 80/20 split — val subset uses val_tf (no augmentation)
full_dataset = NumericalImageFolder(train_dir, transform=train_tf)
num_val      = max(1, int(0.2 * len(full_dataset)))
num_train    = len(full_dataset) - num_val

set_seed(42)
train_subset, val_indices = random_split(full_dataset, [num_train, num_val])

val_base   = NumericalImageFolder(train_dir, transform=val_tf)
val_subset = Subset(val_base, val_indices.indices)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train: {num_train} samples | Val: {num_val} samples')

## 5. Model — EfficientNet-B4

B4 has 19M parameters and a native input size of 380px — significantly stronger than B2 (9M params, 260px) at the cost of ~2× training time.

We load ImageNet-pretrained weights, discard the 1000-class head, and attach a new `1792 → 100` linear layer.

In [ ]:
def build_model(freeze_backbone=False):
    model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features  # 1792 for B4
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, NUM_CLASSES),
    )
    return model.to(device)


# Preview parameter counts
_m = build_model()
total     = sum(p.numel() for p in _m.parameters())
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}')
del _m

## 6. Loss, MixUp, Train/Val Functions

**MixUp:** blends two training images pixel-by-pixel and computes a weighted mix of their losses. Prevents memorizing specific images.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


def mixup_data(x, y):
    """Returns mixed inputs and both label sets with blend ratio lambda."""
    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA) if MIXUP_ALPHA > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def train_one_epoch(model, loader, optimizer):
    """Train for one epoch with MixUp. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, leave=False, desc='Train'):
        images, labels = images.to(device), labels.to(device)
        images, labels_a, labels_b, lam = mixup_data(images, labels)
        optimizer.zero_grad()
        outputs = model(images)
        loss = lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels_a).sum().item()
        total      += labels_a.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model on clean images. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, leave=False, desc='Val  '):
        images, labels = images.to(device), labels.to(device)
        outputs    = model(images)
        loss       = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

## 7. Two-Phase Training Loop

**Phase 1 (epochs 1–5):** Backbone frozen. Only the new head trains at LR=1e-3. Prevents random gradients from corrupting pretrained features.

**Phase 2 (epochs 6–30):** All layers unfrozen. Backbone trains at LR=1e-4 (slow), head at LR=5e-4 (faster). Early stopping halts if val acc doesn't improve for 7 epochs.

In [ ]:
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

history        = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc   = 0.0
best_epoch     = -1
patience_count = 0

model = build_model(freeze_backbone=True)

# ── Phase 1: backbone frozen ──────────────────────────────────────────────────
optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p1, T_max=PHASE1_EPOCHS)

print('=== Phase 1: Warm-up (backbone frozen) ===')
for epoch in range(1, PHASE1_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer_p1)
    val_loss,   val_acc   = evaluate(model, val_loader)
    scheduler_p1.step()
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'[P1] Epoch {epoch:02d}/{PHASE1_EPOCHS} | '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch}, CKPT_PATH)

# ── Phase 2: all layers unfrozen ──────────────────────────────────────────────
for param in model.features.parameters():
    param.requires_grad = True

optimizer_p2 = torch.optim.AdamW([
    {'params': model.features.parameters(),   'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_p2, T_max=PHASE2_EPOCHS)

print('\n=== Phase 2: Fine-tuning (all layers) ===')
for epoch in range(1, PHASE2_EPOCHS + 1):
    global_epoch = PHASE1_EPOCHS + epoch
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer_p2)
    val_loss,   val_acc   = evaluate(model, val_loader)
    scheduler_p2.step()
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'[P2] Epoch {epoch:02d}/{PHASE2_EPOCHS} | '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}')
    if val_acc > best_val_acc:
        best_val_acc   = val_acc
        best_epoch     = global_epoch
        patience_count = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': global_epoch}, CKPT_PATH)
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {global_epoch} '
                  f'(no improvement for {PATIENCE} epochs)')
            break

print(f'\nBest val acc: {best_val_acc:.4f} at epoch {best_epoch}')
print(f'Checkpoint saved to: {CKPT_PATH}')

## 8. Plot Training Curves

In [ ]:
total_epochs = len(history['train_loss'])
epochs_range = range(1, total_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, history['train_loss'], label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   label='Val Loss')
ax1.axvline(PHASE1_EPOCHS + 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase boundary')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()

ax2.plot(epochs_range, history['train_acc'], label='Train Acc')
ax2.plot(epochs_range, history['val_acc'],   label='Val Acc')
ax2.axvline(PHASE1_EPOCHS + 0.5, color='gray', linestyle='--', alpha=0.7, label='Phase boundary')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig('training_curves_v2.png', dpi=150)
plt.show()
print('Saved training_curves_v2.png')

## 9. Load Best Checkpoint & Evaluate

In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

val_loss, val_acc = evaluate(model, val_loader)
print(f'Best model — Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

## 10. Inference with 6-Pass Test-Time Augmentation (TTA)

We run each test image through the model **6 times** with different crops and flips, then average all the raw logit scores before making a final prediction.

| Pass | Transform |
|------|-----------|
| 1 | Standard center crop |
| 2 | Center crop + horizontal flip |
| 3 | Slightly larger crop (zoom out) |
| 4 | Slightly larger crop + horizontal flip |
| 5 | Tight crop (zoom in) |
| 6 | Tight crop + horizontal flip |

More views → more stable prediction → free accuracy with no retraining.

In [ ]:
def _make_tta_tf(resize_factor, hflip=False):
    """Build a single TTA transform at a given scale, with optional hflip."""
    steps = [
        transforms.Resize((int(IMG_SIZE * resize_factor), int(IMG_SIZE * resize_factor))),
        transforms.CenterCrop(IMG_SIZE),
    ]
    if hflip:
        steps.append(transforms.RandomHorizontalFlip(p=1.0))
    steps += [
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
    return transforms.Compose(steps)


# 6 TTA passes: 3 scales × 2 (original + hflip)
tta_transforms = [
    _make_tta_tf(1.15, hflip=False),   # pass 1: standard center crop
    _make_tta_tf(1.15, hflip=True),    # pass 2: standard + hflip
    _make_tta_tf(1.30, hflip=False),   # pass 3: zoomed out
    _make_tta_tf(1.30, hflip=True),    # pass 4: zoomed out + hflip
    _make_tta_tf(1.00, hflip=False),   # pass 5: tight (no padding)
    _make_tta_tf(1.00, hflip=True),    # pass 6: tight + hflip
]

# Collect filenames from the first pass
_ref_dataset = TestDataset(test_dir, transform=tta_transforms[0])
all_filenames = [_ref_dataset[i][1] for i in range(len(_ref_dataset))]

# Accumulate logits across all passes
model.eval()
accum_logits = None

for pass_idx, tf in enumerate(tta_transforms):
    dataset = TestDataset(test_dir, transform=tf)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    pass_logits = []
    with torch.no_grad():
        for images, _ in tqdm(loader, desc=f'TTA pass {pass_idx+1}/{len(tta_transforms)}', leave=False):
            pass_logits.append(model(images.to(device)).cpu())
    pass_logits = torch.cat(pass_logits, dim=0)
    accum_logits = pass_logits if accum_logits is None else accum_logits + pass_logits
    print(f'Pass {pass_idx+1}/{len(tta_transforms)} done')

preds = accum_logits.argmax(dim=1).numpy()
submission = pd.DataFrame({'ID': all_filenames, 'Label': preds})
submission.to_csv('submission.csv', index=False)
print(f'\nSaved submission.csv  (shape: {submission.shape})')
print(f"Label range: {submission['Label'].min()} – {submission['Label'].max()} "
      f"| Unique labels: {submission['Label'].nunique()}")
submission.head(10)

## 11. (Optional) Retrain on Full Dataset

Once hyperparameters are finalized, retrain on all 1079 training samples (no val split) to maximize test performance. Set `RETRAIN_FULL = True` and run this cell.

In [ ]:
RETRAIN_FULL = False

if RETRAIN_FULL:
    set_seed(42)
    full_dataset = NumericalImageFolder(train_dir, transform=train_tf)
    full_loader  = DataLoader(full_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=NUM_WORKERS)

    model_full = build_model(freeze_backbone=True)

    # Phase 1
    opt1 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_full.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS)
    print('=== Full retrain — Phase 1 ===')
    for epoch in range(1, PHASE1_EPOCHS + 1):
        loss, acc = train_one_epoch(model_full, full_loader, opt1)
        sch1.step()
        print(f'  Epoch {epoch}/{PHASE1_EPOCHS} | Loss: {loss:.4f}  Acc: {acc:.4f}')

    # Phase 2
    for param in model_full.features.parameters():
        param.requires_grad = True
    opt2 = torch.optim.AdamW([
        {'params': model_full.features.parameters(),   'lr': 1e-4},
        {'params': model_full.classifier.parameters(), 'lr': 5e-4},
    ], weight_decay=1e-4)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS)
    print('=== Full retrain — Phase 2 ===')
    for epoch in range(1, PHASE2_EPOCHS + 1):
        loss, acc = train_one_epoch(model_full, full_loader, opt2)
        sch2.step()
        print(f'  Epoch {epoch}/{PHASE2_EPOCHS} | Loss: {loss:.4f}  Acc: {acc:.4f}')

    ckpt_full = './checkpoints/final_full_train_v2.pt'
    torch.save({'model_state_dict': model_full.state_dict()}, ckpt_full)
    print(f'Full-data model saved to {ckpt_full}')

    # Run 6-pass TTA on full-data model
    model_full.eval()
    accum_logits_full = None
    for pass_idx, tf in enumerate(tta_transforms):
        dataset = TestDataset(test_dir, transform=tf)
        loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        pass_logits = []
        with torch.no_grad():
            for images, _ in tqdm(loader, desc=f'TTA pass {pass_idx+1}', leave=False):
                pass_logits.append(model_full(images.to(device)).cpu())
        pass_logits = torch.cat(pass_logits, dim=0)
        accum_logits_full = pass_logits if accum_logits_full is None else accum_logits_full + pass_logits

    preds_full = accum_logits_full.argmax(dim=1).numpy()
    sub_full   = pd.DataFrame({'ID': all_filenames, 'Label': preds_full})
    sub_full.to_csv('submission_full.csv', index=False)
    print('Saved submission_full.csv')
else:
    print('RETRAIN_FULL is False — skipping. Set to True when hyperparameters are finalized.')